# ARI711S — Artificial Intelligence: Assignment 1

**Qualification:** Bachelor of Computer Science (Software Development)  
**Course:** Artificial Intelligence (ARI711S) | NQF Level 7  
**Group:** Ninjas 2026  
**Due Date:** 26 April 2026

---

## Group Members

| Name | Student Number |
|------|---------------|
| Member 1 | |
| Member 2 | |
| Member 3 | |
| Member 4 | |
| Member 5 | |

---

## Table of Contents

- [Question 1: Search Algorithms — Warehouse Logistics Robot](#q1)
- [Question 2: Optimisation — Telecom Tower Placement (CSP)](#q2)
- [Question 3: Adversarial Search — Tic-Tac-Toe with Minimax](#q3)

---

<a id='q1'></a>
# Question 1: Search Algorithms — The Warehouse Logistics Bot [25 Marks]

## Background

We implement an **Automated Guided Vehicle (AGV)** navigation system for a warehouse robot. The robot must find the most efficient path from a **Charging Station (A)** to a **Product Bin (B)** in a grid-world environment with shelving units (walls).

## Problem Formulation

| Element | Description |
|---------|-------------|
| **States** | Grid coordinates `(row, col)` |
| **Actions** | Move up, down, left, right to non-wall adjacent cells |
| **Initial State** | Position marked `A` |
| **Goal State** | Position marked `B` |
| **Heuristic** | Euclidean Distance: $h(n) = \sqrt{(x_1-x_2)^2 + (y_1-y_2)^2}$ |

## Algorithms

- **Greedy Best-First Search:** Priority = `h(n)` only
- **A\* Search:** Priority = `f(n) = g(n) + h(n)` (path cost + heuristic)

---

In [ ]:
# ── Q1 Imports ─────────────────────────────────────────────────────────────
import heapq
import math
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

### Q1.1 — Node Class

Stores the search state, its parent (for path reconstruction), the action taken, and the cumulative path cost `g(n)`.

In [ ]:
class Node:
    """
    Represents a single state in the search tree.

    Attributes:
        state  (tuple): Grid coordinates (row, col).
        parent (Node) : The node from which this node was reached.
        action (str)  : Movement taken to reach this node.
        g      (int)  : Path cost from the start node to this node.
    """

    def __init__(self, state, parent=None, action=None, g=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.g = g

    def __lt__(self, other):
        return self.g < other.g

    def __eq__(self, other):
        return isinstance(other, Node) and self.state == other.state

    def __hash__(self):
        return hash(self.state)

    def path(self):
        """Reconstruct the path from start to this node by tracing parents."""
        nodes = []
        current = self
        while current is not None:
            nodes.append(current)
            current = current.parent
        return list(reversed(nodes))

### Q1.2 — Warehouse Class

Reads the `.txt` layout, locates A and B, stores shelving positions, provides valid neighbours, runs search, and visualises results.

In [ ]:
class Warehouse:
    """
    Models the warehouse environment.

    Layout file conventions:
        '#' = shelving unit (wall)
        'A' = charging station (start)
        'B' = product bin (goal)
        ' ' = open aisle
    """

    def __init__(self, filename):
        self.walls = set()
        self.start = None
        self.goal = None
        self.grid = []

        with open(filename, 'r') as f:
            lines = f.read().splitlines()

        self.rows = len(lines)
        self.cols = max(len(line) for line in lines)

        for row, line in enumerate(lines):
            row_data = []
            for col, char in enumerate(line):
                row_data.append(char)
                if char == '#':
                    self.walls.add((row, col))
                elif char == 'A':
                    self.start = (row, col)
                elif char == 'B':
                    self.goal = (row, col)
            while len(row_data) < self.cols:
                row_data.append(' ')
            self.grid.append(row_data)

        if self.start is None:
            raise ValueError("No start position 'A' found in the warehouse layout.")
        if self.goal is None:
            raise ValueError("No goal position 'B' found in the warehouse layout.")

        print(f"Warehouse loaded: {self.rows} rows x {self.cols} cols")
        print(f"Charging Station (A): {self.start}")
        print(f"Product Bin     (B): {self.goal}")
        print(f"Shelving units  (#): {len(self.walls)} cells")

    def heuristic(self, state):
        """Euclidean distance from state to the goal."""
        r1, c1 = state
        r2, c2 = self.goal
        return math.sqrt((r1 - r2) ** 2 + (c1 - c2) ** 2)

    def neighbors(self, state):
        """Return valid adjacent cells (up, down, left, right)."""
        row, col = state
        directions = [
            ('up',    row - 1, col),
            ('down',  row + 1, col),
            ('left',  row,     col - 1),
            ('right', row,     col + 1),
        ]
        return [
            (action, (r, c))
            for action, r, c in directions
            if 0 <= r < self.rows and 0 <= c < self.cols and (r, c) not in self.walls
        ]

    def solve(self, algorithm="astar"):
        """
        Search for a path from A to B.

        Parameters:
            algorithm (str): 'greedy' or 'astar'

        Returns:
            path (list[Node]), explored (set)
        """
        algorithm = algorithm.lower()
        start_node = Node(state=self.start, g=0)
        counter = 0

        if algorithm == "greedy":
            priority = self.heuristic(self.start)
        else:
            priority = self.heuristic(self.start)

        frontier = []
        heapq.heappush(frontier, (priority, counter, start_node))

        best_g = {self.start: 0}
        explored = set()

        while frontier:
            _, _, current = heapq.heappop(frontier)

            if current.state == self.goal:
                return current.path(), explored

            if current.state in explored:
                continue
            explored.add(current.state)

            for action, neighbor_state in self.neighbors(current.state):
                new_g = current.g + 1
                if neighbor_state not in best_g or new_g < best_g[neighbor_state]:
                    best_g[neighbor_state] = new_g
                    child = Node(state=neighbor_state, parent=current,
                                 action=action, g=new_g)
                    h = self.heuristic(neighbor_state)
                    priority = h if algorithm == "greedy" else new_g + h
                    counter += 1
                    heapq.heappush(frontier, (priority, counter, child))

        raise ValueError("No path found from A to B.")

    def visualise(self, path, explored, algorithm_name, save_as="warehouse_path.png"):
        """Render the warehouse grid and save the image."""
        COLOR_OPEN     = [1.00, 1.00, 1.00]
        COLOR_WALL     = [0.20, 0.20, 0.20]
        COLOR_EXPLORED = [0.75, 0.90, 1.00]
        COLOR_PATH     = [0.00, 0.85, 0.85]
        COLOR_START    = [0.00, 0.80, 0.20]
        COLOR_GOAL     = [0.90, 0.15, 0.15]

        image = np.ones((self.rows, self.cols, 3))
        for r in range(self.rows):
            for c in range(self.cols):
                image[r, c] = COLOR_WALL if (r, c) in self.walls else COLOR_OPEN

        for state in explored:
            if state not in (self.start, self.goal):
                image[state[0], state[1]] = COLOR_EXPLORED

        for node in path:
            if node.state not in (self.start, self.goal):
                image[node.state[0], node.state[1]] = COLOR_PATH

        image[self.start[0], self.start[1]] = COLOR_START
        image[self.goal[0],  self.goal[1]]  = COLOR_GOAL

        fig, ax = plt.subplots(figsize=(max(8, self.cols // 2), max(6, self.rows // 2)))
        ax.imshow(image, interpolation='nearest')
        ax.text(self.start[1], self.start[0], 'A',
                ha='center', va='center', fontsize=9, fontweight='bold', color='white')
        ax.text(self.goal[1], self.goal[0], 'B',
                ha='center', va='center', fontsize=9, fontweight='bold', color='white')
        ax.set_xticks(np.arange(-0.5, self.cols, 1), minor=True)
        ax.set_yticks(np.arange(-0.5, self.rows, 1), minor=True)
        ax.grid(which='minor', color='#cccccc', linewidth=0.4)
        ax.tick_params(which='both', bottom=False, left=False,
                       labelbottom=False, labelleft=False)
        legend_items = [
            mpatches.Patch(color=COLOR_START,    label='Charging Station (A)'),
            mpatches.Patch(color=COLOR_GOAL,     label='Product Bin (B)'),
            mpatches.Patch(color=COLOR_PATH,     label='Optimal Path'),
            mpatches.Patch(color=COLOR_EXPLORED, label='Explored (not on path)'),
            mpatches.Patch(color=COLOR_WALL,     label='Shelving Unit (Wall)'),
        ]
        ax.legend(handles=legend_items, loc='upper right',
                  fontsize=8, framealpha=0.9, bbox_to_anchor=(1.35, 1.0))
        ax.set_title(f"Warehouse Robot — {algorithm_name}\n"
                     f"Path length: {len(path)-1} steps  |  "
                     f"States explored: {len(explored)}", fontsize=11)
        plt.tight_layout()
        plt.savefig(save_as, dpi=150, bbox_inches='tight')
        plt.show()
        print(f"Saved: '{save_as}'")

### Q1.3 — Load Warehouse and Run Both Algorithms

In [ ]:
# Load warehouse — replace path if your warehouse.txt is in a different location
warehouse = Warehouse('question1/warehouse.txt')

In [ ]:
print("=" * 50)
print("GREEDY BEST-FIRST SEARCH")
print("=" * 50)
greedy_path, greedy_explored = warehouse.solve(algorithm="greedy")
print(f"Path steps: {len(greedy_path)-1}  |  States explored: {len(greedy_explored)}")

warehouse.visualise(greedy_path, greedy_explored,
                    "Greedy Best-First Search",
                    save_as="warehouse_path_greedy.png")

In [ ]:
print("=" * 50)
print("A* SEARCH")
print("=" * 50)
astar_path, astar_explored = warehouse.solve(algorithm="astar")
print(f"Path steps: {len(astar_path)-1}  |  States explored: {len(astar_explored)}")

warehouse.visualise(astar_path, astar_explored,
                    "A* Search",
                    save_as="warehouse_path.png")

In [ ]:
# Side-by-side comparison
def build_image(w, path, explored):
    image = np.ones((w.rows, w.cols, 3))
    for r in range(w.rows):
        for c in range(w.cols):
            image[r, c] = [0.2,0.2,0.2] if (r,c) in w.walls else [1,1,1]
    for s in explored:
        if s not in (w.start, w.goal):
            image[s[0],s[1]] = [0.75,0.90,1.00]
    for n in path:
        if n.state not in (w.start, w.goal):
            image[n.state[0],n.state[1]] = [0,0.85,0.85]
    image[w.start[0],w.start[1]] = [0,0.80,0.20]
    image[w.goal[0],w.goal[1]]   = [0.90,0.15,0.15]
    return image

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
for ax, (name, path, explored) in zip(axes, [
    ("Greedy Best-First Search", greedy_path, greedy_explored),
    ("A* Search",               astar_path,  astar_explored),
]):
    ax.imshow(build_image(warehouse, path, explored), interpolation='nearest')
    ax.set_title(f"{name}\nPath: {len(path)-1} steps  |  Explored: {len(explored)}", fontsize=11)
    ax.text(warehouse.start[1], warehouse.start[0], 'A', ha='center', va='center',
            fontsize=9, fontweight='bold', color='white')
    ax.text(warehouse.goal[1], warehouse.goal[0], 'B', ha='center', va='center',
            fontsize=9, fontweight='bold', color='white')
    ax.set_xticks([]); ax.set_yticks([])

legend_items = [
    mpatches.Patch(color=[0,0.80,0.20],  label='Charging Station (A)'),
    mpatches.Patch(color=[0.90,0.15,0.15], label='Product Bin (B)'),
    mpatches.Patch(color=[0,0.85,0.85],  label='Optimal Path'),
    mpatches.Patch(color=[0.75,0.90,1.00], label='Explored'),
    mpatches.Patch(color=[0.2,0.2,0.2],  label='Shelving Unit'),
]
fig.legend(handles=legend_items, loc='lower center', ncol=5, fontsize=9)
plt.suptitle("Warehouse Robot — Algorithm Comparison", fontsize=13)
plt.tight_layout()
plt.savefig("warehouse_comparison.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 'warehouse_comparison.png'")

### Q1 — Analysis

| Metric | Greedy Best-First | A* |
|--------|------------------|----|
| **Priority** | `h(n)` only | `g(n) + h(n)` |
| **Optimal?** | Not guaranteed | Guaranteed |
| **Complete?** | Not complete | Complete |
| **Heuristic** | Euclidean distance | Euclidean distance |

**A\*** is preferred in warehouse logistics because it guarantees the shortest path, directly reducing the AGV's travel time and energy use. The Euclidean heuristic is **admissible** (never overestimates), ensuring A\* optimality.

---

<a id='q2'></a>
# Question 2: Optimisation — Telecommunication Tower Placement [25 Marks]

## Background

MTC must place **8 signal boosters** on a **10×10 grid** satisfying three constraints:
1. No two towers share the same row or column
2. No two towers are in adjacent cells (including diagonals)
3. No tower on a mountain (prohibited) cell

## Solution Strategy

**Backtracking** with:
- **MRV (Minimum Remaining Values):** Always assign the most constrained tower first
- **Forward Checking:** Prune domains after each placement

---

In [ ]:
# ── Q2 Imports ─────────────────────────────────────────────────────────────
import copy as _copy
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

In [ ]:
class Telecom_CSP_Solver:
    """
    Solves the MTC 5G tower placement problem as a CSP.

    Variables  : 8 towers (indexed 0-7)
    Domain     : All valid (non-mountain) cells in a 10x10 grid
    Constraints:
        1. No two towers share a row or column (signal overlap)
        2. No two towers adjacent including diagonals (interference buffer)
        3. No tower on mountain cells (terrain obstruction)
    """

    GRID_SIZE  = 10
    NUM_TOWERS = 8

    def __init__(self, mountains=None):
        self.mountains = set(mountains) if mountains else set()
        all_cells = [(r, c) for r in range(self.GRID_SIZE)
                            for c in range(self.GRID_SIZE)]
        initial_domain = [cell for cell in all_cells
                          if cell not in self.mountains]
        self.domains    = [set(initial_domain) for _ in range(self.NUM_TOWERS)]
        self.solution   = None
        self.node_count = 0
        print(f"Grid: {self.GRID_SIZE}x{self.GRID_SIZE} | Towers: {self.NUM_TOWERS} | "
              f"Mountains: {len(self.mountains)} | Domain: {len(initial_domain)} cells")

    def is_consistent(self, assignment, cell):
        """Return True if placing a tower at cell satisfies all constraints."""
        r_new, c_new = cell
        for _, (r_p, c_p) in assignment.items():
            if r_new == r_p or c_new == c_p:
                return False
            if abs(r_new - r_p) <= 1 and abs(c_new - c_p) <= 1:
                return False
        return True

    def select_unassigned_variable(self, assignment, domains):
        """MRV: choose the unassigned tower with the smallest domain."""
        unassigned = [i for i in range(self.NUM_TOWERS) if i not in assignment]
        return min(unassigned, key=lambda i: len(domains[i]))

    def forward_check(self, assignment, tower_idx, cell, domains):
        """Prune domains of unassigned towers after placing tower_idx at cell."""
        r_p, c_p = cell
        for other in range(self.NUM_TOWERS):
            if other in assignment:
                continue
            to_remove = set()
            for candidate in domains[other]:
                r_c, c_c = candidate
                if r_c == r_p or c_c == c_p:
                    to_remove.add(candidate)
                elif abs(r_c - r_p) <= 1 and abs(c_c - c_p) <= 1:
                    to_remove.add(candidate)
            domains[other] -= to_remove
            if not domains[other]:
                return False
        return True

    def backtrack(self, assignment, domains):
        """Core recursive backtracking search."""
        self.node_count += 1
        if len(assignment) == self.NUM_TOWERS:
            return assignment
        tower = self.select_unassigned_variable(assignment, domains)
        for cell in sorted(domains[tower]):
            if self.is_consistent(assignment, cell):
                assignment[tower] = cell
                new_domains = _copy.deepcopy(domains)
                new_domains[tower] = {cell}
                if self.forward_check(assignment, tower, cell, new_domains):
                    result = self.backtrack(assignment, new_domains)
                    if result is not None:
                        return result
                del assignment[tower]
        return None

    def solve(self):
        """Initiate backtracking from an empty assignment."""
        print("Searching...")
        self.node_count = 0
        result = self.backtrack({}, _copy.deepcopy(self.domains))
        if result is not None:
            self.solution = result
            print(f"Solution found after {self.node_count} calls.")
            for t in sorted(result):
                print(f"  T{t+1}: {result[t]}")
        else:
            print("No solution found.")
        return result

    def visualise(self, assignment, title="MTC Tower Placement", save_as=None):
        """Plot the 10x10 grid showing towers and mountains."""
        if assignment is None:
            return
        tower_cells = set(assignment.values())
        tower_idx_map = {v: k for k, v in assignment.items()}
        COLOR_OPEN     = [0.94, 0.97, 0.94]
        COLOR_MOUNTAIN = [0.60, 0.40, 0.20]
        COLOR_TOWER    = [0.18, 0.45, 0.71]
        image = np.ones((self.GRID_SIZE, self.GRID_SIZE, 3))
        for r in range(self.GRID_SIZE):
            for c in range(self.GRID_SIZE):
                if (r, c) in self.mountains:
                    image[r, c] = COLOR_MOUNTAIN
                elif (r, c) in tower_cells:
                    image[r, c] = COLOR_TOWER
                else:
                    image[r, c] = COLOR_OPEN
        fig, ax = plt.subplots(figsize=(7, 7))
        ax.imshow(image, interpolation='nearest')
        ax.set_xticks(np.arange(-0.5, self.GRID_SIZE, 1), minor=True)
        ax.set_yticks(np.arange(-0.5, self.GRID_SIZE, 1), minor=True)
        ax.grid(which='minor', color='#888888', linewidth=0.8)
        ax.set_xticks(range(self.GRID_SIZE)); ax.set_yticks(range(self.GRID_SIZE))
        ax.set_xticklabels(range(self.GRID_SIZE), fontsize=8)
        ax.set_yticklabels(range(self.GRID_SIZE), fontsize=8)
        ax.tick_params(which='minor', length=0)
        ax.set_xlabel("Column", fontsize=10); ax.set_ylabel("Row", fontsize=10)
        for r in range(self.GRID_SIZE):
            for c in range(self.GRID_SIZE):
                if (r, c) in tower_cells:
                    ax.text(c, r, f"T{tower_idx_map[(r,c)]+1}",
                            ha='center', va='center', fontsize=9,
                            fontweight='bold', color='white')
                elif (r, c) in self.mountains:
                    ax.text(c, r, 'M', ha='center', va='center',
                            fontsize=9, fontweight='bold', color='white')
        legend_items = [
            mpatches.Patch(color=COLOR_TOWER,    label='Signal Booster (T)'),
            mpatches.Patch(color=COLOR_MOUNTAIN, label='Mountain (M)'),
            mpatches.Patch(color=COLOR_OPEN,     label='Open Cell'),
        ]
        ax.legend(handles=legend_items, loc='upper right',
                  fontsize=8, framealpha=0.9, bbox_to_anchor=(1.38, 1.0))
        ax.set_title(title, fontsize=12, pad=12)
        plt.tight_layout()
        if save_as:
            plt.savefig(save_as, dpi=150, bbox_inches='tight')
            print(f"Saved: '{save_as}'")
        plt.show()

    def verify_solution(self, assignment):
        """Independently verify all constraints are satisfied."""
        if assignment is None:
            return False
        cells = list(assignment.values())
        passed = True
        for idx, cell in assignment.items():
            if cell in self.mountains:
                print(f"  FAIL: T{idx+1} on mountain {cell}")
                passed = False
        for i in range(len(cells)):
            for j in range(i + 1, len(cells)):
                r1, c1 = cells[i]; r2, c2 = cells[j]
                t1 = list(assignment.keys())[i] + 1
                t2 = list(assignment.keys())[j] + 1
                if r1 == r2:
                    print(f"  FAIL: T{t1} and T{t2} share row {r1}")
                    passed = False
                if c1 == c2:
                    print(f"  FAIL: T{t1} and T{t2} share col {c1}")
                    passed = False
                if abs(r1-r2) <= 1 and abs(c1-c2) <= 1:
                    print(f"  FAIL: T{t1} and T{t2} are adjacent")
                    passed = False
        if passed:
            print("  All constraints satisfied — VALID")
        return passed

### Q2.1 — Level 1: Coastal Region (Easy)

In [ ]:
print("=" * 50); print("LEVEL 1 — COASTAL (EASY)"); print("=" * 50)
solver1 = Telecom_CSP_Solver(mountains=[(0,0),(1,1),(9,9)])
sol1 = solver1.solve()
print("\nVerification:"); solver1.verify_solution(sol1)
solver1.visualise(sol1, "Level 1 — Coastal (Easy)", save_as="towers_level1_coastal.png")

### Q2.2 — Level 2: Highlands Region (Medium)

In [ ]:
print("=" * 50); print("LEVEL 2 — HIGHLANDS (MEDIUM)"); print("=" * 50)
solver2 = Telecom_CSP_Solver(mountains=[(2,2),(2,3),(3,2),(3,3),(7,8),(8,7),(8,8)])
sol2 = solver2.solve()
print("\nVerification:"); solver2.verify_solution(sol2)
solver2.visualise(sol2, "Level 2 — Highlands (Medium)", save_as="towers_level2_highlands.png")

### Q2.3 — Level 3: Brandberg Region (Hard)

In [ ]:
print("=" * 50); print("LEVEL 3 — BRANDBERG (HARD)"); print("=" * 50)
solver3 = Telecom_CSP_Solver(
    mountains=[(0,5),(1,5),(2,5),(3,5),(4,5),(5,0),(5,1),(5,2),(5,3),(5,4)])
sol3 = solver3.solve()
print("\nVerification:"); solver3.verify_solution(sol3)
solver3.visualise(sol3, "Level 3 — Brandberg (Hard)", save_as="towers_level3_brandberg.png")

### Q2 — Analysis

| Component | Purpose |
|-----------|---------|
| `backtrack()` | Core DFS — places one tower per call, undoes failed choices |
| `MRV` | Front-loads the hardest decisions, catching failures early |
| `forward_check()` | Removes newly impossible cells after each placement |
| `is_consistent()` | Row/column + Chebyshev-distance check before each placement |

| Level | Mountains | Key Challenge |
|-------|-----------|---------------|
| 1 — Coastal | 3 scattered | Baseline logic |
| 2 — Highlands | 7 clustered | Navigating around 2×2 blocks |
| 3 — Brandberg | 10 (L-wall) | Grid effectively split; towers forced into tight clusters |

---

<a id='q3'></a>
# Question 3: Adversarial Search — Tic-Tac-Toe with Minimax [25 Marks]

## Background

We implement an AI agent that plays **Tic-Tac-Toe optimally** using the **Minimax algorithm with Alpha-Beta Pruning**. The AI is unbeatable — it will always win or draw, never lose.

| Element | Description |
|---------|-------------|
| **States** | 3×3 game boards |
| **Players** | X (maximiser, +1) and O (minimiser, −1) |
| **Terminal** | Board full, or three in a row |
| **Utility** | +1 (X wins), −1 (O wins), 0 (tie) |

---

In [ ]:
# ── Q3 Imports ─────────────────────────────────────────────────────────────
import copy as _copy2
import math
import matplotlib.pyplot as plt

X_TOKEN = "X"
O_TOKEN = "O"
EMPTY   = None

def empty_board():
    return [[EMPTY]*3 for _ in range(3)]

In [ ]:
def player(board):
    """Return whose turn it is: X or O. X goes first."""
    x = sum(row.count(X_TOKEN) for row in board)
    o = sum(row.count(O_TOKEN) for row in board)
    return X_TOKEN if x == o else O_TOKEN

def actions(board):
    """Return set of all legal moves (i, j) on the board."""
    return {(i,j) for i in range(3) for j in range(3) if board[i][j] == EMPTY}

def result(board, action):
    """Return new board after current player makes action. Does not modify original."""
    i, j = action
    if board[i][j] != EMPTY:
        raise ValueError(f"Cell {action} is already occupied by '{board[i][j]}'.")
    new_board = _copy2.deepcopy(board)
    new_board[i][j] = player(board)
    return new_board

def winner(board):
    """Return 'X', 'O', or None."""
    lines = [
        [(0,0),(0,1),(0,2)], [(1,0),(1,1),(1,2)], [(2,0),(2,1),(2,2)],
        [(0,0),(1,0),(2,0)], [(0,1),(1,1),(2,1)], [(0,2),(1,2),(2,2)],
        [(0,0),(1,1),(2,2)], [(0,2),(1,1),(2,0)],
    ]
    for line in lines:
        vals = [board[r][c] for r,c in line]
        if vals[0] is not None and vals[0] == vals[1] == vals[2]:
            return vals[0]
    return None

def terminal(board):
    """Return True if the game is over."""
    if winner(board) is not None:
        return True
    return all(board[i][j] != EMPTY for i in range(3) for j in range(3))

def utility(board):
    """Return 1, -1, or 0 for a terminal board."""
    w = winner(board)
    return 1 if w == X_TOKEN else (-1 if w == O_TOKEN else 0)

In [ ]:
def max_value(board, alpha, beta):
    """MAX (X) — return best achievable utility with alpha-beta pruning."""
    if terminal(board):
        return utility(board)
    v = -math.inf
    for action in actions(board):
        v = max(v, min_value(result(board, action), alpha, beta))
        alpha = max(alpha, v)
        if alpha >= beta:
            break
    return v

def min_value(board, alpha, beta):
    """MIN (O) — return best achievable utility with alpha-beta pruning."""
    if terminal(board):
        return utility(board)
    v = math.inf
    for action in actions(board):
        v = min(v, max_value(result(board, action), alpha, beta))
        beta = min(beta, v)
        if alpha >= beta:
            break
    return v

def minimax(board):
    """Return optimal (i, j) action for the current player, or None if terminal."""
    if terminal(board):
        return None
    cur = player(board)
    best_action = None
    if cur == X_TOKEN:
        best_score = -math.inf
        for action in actions(board):
            score = min_value(result(board, action), -math.inf, math.inf)
            if score > best_score:
                best_score = score; best_action = action
    else:
        best_score = math.inf
        for action in actions(board):
            score = max_value(result(board, action), -math.inf, math.inf)
            if score < best_score:
                best_score = score; best_action = action
    return best_action

### Q3.1 — Test Cases

In [ ]:
def print_board(board, label=""):
    sym = {X_TOKEN:'X', O_TOKEN:'O', EMPTY:'.'}
    if label: print(f"\n{label}")
    print("  0 1 2")
    for i, row in enumerate(board):
        print(f"{i} " + " ".join(sym[c] for c in row))

# player()
b = empty_board()
print(f"Empty board → player: {player(b)} (expected: X)")
print(f"Empty board → actions: {len(actions(b))} (expected: 9)")

# winner()
b_xwin = [[X_TOKEN,X_TOKEN,X_TOKEN],[O_TOKEN,O_TOKEN,None],[None,None,None]]
print(f"X top row → winner: {winner(b_xwin)} (expected: X)")

b_owin = [[O_TOKEN,X_TOKEN,X_TOKEN],[O_TOKEN,X_TOKEN,None],[O_TOKEN,None,None]]
print(f"O left col → winner: {winner(b_owin)} (expected: O)")

b_tie = [[X_TOKEN,O_TOKEN,X_TOKEN],[X_TOKEN,X_TOKEN,O_TOKEN],[O_TOKEN,X_TOKEN,O_TOKEN]]
print(f"Tie → winner: {winner(b_tie)} (expected: None), utility: {utility(b_tie)} (expected: 0)")

# result() immutability
orig = empty_board()
new = result(orig, (0,0))
print(f"Original[0][0]: {orig[0][0]} (expected: None)  New[0][0]: {new[0][0]} (expected: X)")

# minimax() forced win
b_mustwin = [[X_TOKEN,None,X_TOKEN],[O_TOKEN,O_TOKEN,None],[None,None,None]]
move = minimax(b_mustwin)
print_board(b_mustwin, "X must win immediately:")
print(f"minimax → {move} (expected: (0,1))")

# minimax() forced block
b_block = [[X_TOKEN,X_TOKEN,None],[O_TOKEN,None,None],[None,None,None]]
move2 = minimax(b_block)
print_board(b_block, "O must block at (0,2):")
print(f"minimax → {move2} (expected: (0,2))")

### Q3.2 — Game Simulations

In [ ]:
def simulate_game(x_strategy, o_strategy, label):
    board = empty_board()
    history = [_copy2.deepcopy(board)]
    moves = []
    print(f"\n{'='*45}\n{label}\n{'='*45}")
    while not terminal(board):
        cur = player(board)
        action = x_strategy(board) if cur == X_TOKEN else o_strategy(board)
        board = result(board, action)
        moves.append((cur, action))
        history.append(_copy2.deepcopy(board))
    w = winner(board)
    print(f"Result: {w + ' WINS' if w else 'TIE'}")
    return history, moves

def naive(board):
    for i in range(3):
        for j in range(3):
            if board[i][j] == EMPTY: return (i,j)

h1, m1 = simulate_game(minimax, minimax, "Game 1: AI (X) vs AI (O)")
h2, m2 = simulate_game(minimax, naive,   "Game 2: AI (X) vs Naive (O)")
h3, m3 = simulate_game(naive,   minimax, "Game 3: Naive (X) vs AI (O)")

In [ ]:
def plot_board(board, title="", highlight=None, ax=None):
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(3.5, 3.5))
    ax.set_xlim(0,3); ax.set_ylim(0,3); ax.set_aspect('equal'); ax.axis('off')
    for x in [1,2]: ax.axvline(x, color='#333', lw=2)
    for y in [1,2]: ax.axhline(y, color='#333', lw=2)
    if highlight:
        hr, hc = highlight
        ax.add_patch(plt.Rectangle((hc, 2-hr), 1, 1, color='#fffacd', zorder=0))
    for i in range(3):
        for j in range(3):
            cx, cy = j+0.5, (2-i)+0.5
            if board[i][j] == X_TOKEN:
                ax.plot([cx-.3,cx+.3],[cy-.3,cy+.3], color='#e74c3c', lw=4)
                ax.plot([cx+.3,cx-.3],[cy-.3,cy+.3], color='#e74c3c', lw=4)
            elif board[i][j] == O_TOKEN:
                ax.add_patch(plt.Circle((cx,cy), 0.32, color='#2980b9', fill=False, lw=4))
    if title: ax.set_title(title, fontsize=9, pad=5)
    if standalone: plt.tight_layout(); plt.show()

def plot_game(history, moves, title):
    n = len(history)
    fig, axes = plt.subplots(1, n, figsize=(3*n, 3.5))
    if n == 1: axes = [axes]
    for idx, (ax, board) in enumerate(zip(axes, history)):
        hl = moves[idx-1][1] if idx > 0 else None
        lbl = "Start" if idx == 0 else f"Move {idx}: {moves[idx-1][0]}→{moves[idx-1][1]}"
        if idx == len(history)-1:
            w = winner(board)
            lbl += f"\n{'✓ '+w+' WINS' if w else '= TIE'}"
        plot_board(board, title=lbl, highlight=hl, ax=ax)
    plt.suptitle(title, fontsize=11, y=1.04)
    plt.tight_layout()
    plt.savefig(title.replace(' ','_').replace(':','')[:40]+".png", dpi=120, bbox_inches='tight')
    plt.show()

plot_game(h1, m1, "Game 1: AI(X) vs AI(O) — Expected Tie")
plot_game(h2, m2, "Game 2: AI(X) vs Naive(O)")
plot_game(h3, m3, "Game 3: Naive(X) vs AI(O)")

### Q3 — Analysis

| Function | Behaviour |
|----------|-----------|
| `player()` | Equal X/O count → X's turn; otherwise O |
| `actions()` | All empty cells as `(i,j)` set |
| `result()` | Deep copy → apply move → return new board |
| `winner()` | Checks all 8 lines (3 rows, 3 cols, 2 diagonals) |
| `terminal()` | True if winner found or board full |
| `utility()` | +1 (X), −1 (O), 0 (tie) |
| `minimax()` | Optimal move via full tree search + alpha-beta pruning |

**Why the AI is unbeatable:** Minimax explores all possible future states and guarantees the best outcome given optimal play. When both players play optimally, Tic-Tac-Toe always ends in a **tie** — a known game-theoretic result.

**Alpha-Beta Pruning** reduces explored nodes from O(b^d) to O(b^(d/2)) in the best case by skipping branches that cannot improve on the current best, without affecting the result.

---

## References

- Russell, S. & Norvig, P. (2021). *Artificial Intelligence: A Modern Approach* (4th ed.)
- Lecture slides and lab materials — ARI711S, NUST 2026